In [ ]:

PROJECT_ID = "outwaterc-demo1" # @param {"type":"string","placeholder":"outwaterc-demo1"}
DATASET_ID = "demo_loans" # @param {"type":"string","placeholder":"bt-demo-1"}
TABLE_ID = "loan_records" # @param {"type":"string","placeholder":"loan_records"}

In [ ]:
import pandas as pd
from faker import Faker
import random
from google.cloud import bigquery
from google.api_core import exceptions

# 1. Initialize Faker and BigQuery Client
fake = Faker()
client = bigquery.Client()

# Configuration
TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

# 2. Function to generate a loan-specific note
def generate_loan_note():
    loan_types = ['Mortgage', 'Personal Loan', 'Auto Loan', 'SME Credit']
    status = ['approved', 'under review', 'pending documents', 'rejected']

    note_template = (
        f"Customer expressed interest in a {random.choice(loan_types)}. "
        f"Current status is {random.choice(status)}. "
        f"Notes: {fake.paragraph(nb_sentences=3)} "
        f"Follow up scheduled for {fake.future_date()}."
    )
    return note_template

# 3. Create synthetic data
data = [
    {
      #  "note_id": fake.uuid4(),
        "customer_name": fake.name(),
        "customer_email": fake.email(),
        "customer_phone": fake.phone_number(),
        "customer_address": fake.address(),
        "customer_city": fake.city(),
        "customer_state": fake.state(),
        "customer_zip": fake.zipcode(),
        "loan_id": fake.bothify(text='############'),
        "loan_type": random.choice(['Mortgage', 'Personal Loan', 'Auto Loan', 'SME Credit']),
        "loan_amount": round(random.uniform(1000, 100000), 2),
        "loan_term": random.randint(1, 36),
        "loan_date": fake.date_between(start_date='-1y', end_date='today'),
        "loan_status": random.choice(['approved', 'under review', 'pending documents', 'rejected']),
        "loan_note_text": generate_loan_note()
    }

    for _ in range(100) # Generate 100 rows
]

df = pd.DataFrame(data)

# 4. Load to BigQuery (Creates table if it doesn't exist)
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND", # Options: WRITE_TRUNCATE, WRITE_APPEND, WRITE_EMPTY
    autodetect=True,                  # Automatically infer schema from DataFrame
)

try:
    # This will create the table if it doesn't exist due to the 'autodetect' and 'LoadJob'
    job = client.load_table_from_dataframe(df, TABLE_REF, job_config=job_config)
    job.result()  # Wait for the job to complete
    print(f"Successfully loaded {len(df)} rows to {TABLE_REF}.")
except Exception as e:
    print(f"Error: {e}")

Successfully loaded 100 rows to outwaterc-demo1.demo_loans.loan_records.
